# Graph Convolutional Network (GCN) do Zero

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

GCNs generalizam convoluções para grafos calculando a média das features dos vizinhos de cada nó. Com normalização $\tilde{A} = D^{-1/2}(A + I)D^{-1/2}$, a atualização é $H^{(l+1)} = \sigma(\tilde{A}\,H^{(l)}\,W^{(l)})$.


## Formulação Matemática

$$H^{(l+1)} = \sigma\!\left(D^{-\tfrac{1}{2}}(A + I) D^{-\tfrac{1}{2}}\,H^{(l)}\,W^{(l)}\right)$$

Adicionar $I$ inclui self-loops; a normalização por $D^{-1/2}$ evita que nós com grau alto dominem.


## Implementação


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
def normalize_adj(A):
    A = A + torch.eye(A.size(0), device=A.device)  # add self-loops
    d = A.sum(1)
    d_inv_sqrt = d.pow(-0.5)
    D = torch.diag(d_inv_sqrt)
    return D @ A @ D

class GCNLayer(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.W = nn.Linear(in_f, out_f, bias=False)
    def forward(self, X, A_hat):
        return A_hat @ self.W(X)

class GCN(nn.Module):
    def __init__(self, in_f, hid, classes):
        super().__init__()
        self.l1 = GCNLayer(in_f, hid)
        self.l2 = GCNLayer(hid, classes)
    def forward(self, X, A_hat):
        h = F.relu(self.l1(X, A_hat))
        return self.l2(h, A_hat)


## Experimento


In [ ]:
# Toy graph: Karate-club-like with 6 nodes
torch.manual_seed(0)
N, F_in, C = 6, 4, 2
X = torch.randn(N, F_in)
A = torch.tensor([
    [0,1,1,0,0,0],
    [1,0,1,0,0,0],
    [1,1,0,1,0,0],
    [0,0,1,0,1,1],
    [0,0,0,1,0,1],
    [0,0,0,1,1,0],
], dtype=torch.float)
A_hat = normalize_adj(A)
y = torch.tensor([0,0,0,1,1,1])

model = GCN(F_in, 8, C)
opt = torch.optim.Adam(model.parameters(), lr=0.05)
for epoch in range(80):
    logits = model(X, A_hat)
    loss = F.cross_entropy(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()
print('predictions:', logits.argmax(1).tolist())
print('targets    :', y.tolist())


## Discussão

- Para grafos reais use adjacências esparsas — matrizes densas explodem acima de alguns milhares de nós.
- Duas camadas já atingem vizinhos a 2 hops; over-smoothing aparece além de 3–4 camadas.
- Variantes mais expressivas: GraphSAGE (amostragem de vizinhos), GAT (pesos de atenção), GIN (agregador por soma).


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
